In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [39]:
df = pd.read_csv("preprocessed_loan_data.csv")

In [40]:
df

,age,gender,marital_status,education_level,annual_income,monthly_income,employment_status,debt_to_income_ratio,credit_score,loan_amount,...,loan_term,installment,grade_subgrade,num_of_open_accounts,total_credit_limit,current_balance,delinquency_history,public_records,num_of_delinquencies,loan_paid_back
0,59,1,1,2,24240.19,2020.02,0,0.074,743,17173.72,...,36,581.88,9,7,40833.47,24302.07,1,0,1,1
1,72,0,1,0,20172.98,1681.08,0,0.219,531,22663.89,...,60,573.17,25,5,27968.01,10803.01,1,0,3,1
2,49,0,2,1,26181.80,2181.82,0,0.234,779,3631.36,...,60,76.32,8,2,15502.25,4505.44,0,0,0,1
3,35,0,2,1,11873.84,989.49,0,0.264,809,14939.23,...,36,468.07,4,7,18157.79,5525.63,4,0,5,1
4,63,2,2,3,25326.44,2110.54,0,0.260,663,16551.71,...,60,395.50,19,1,17467.56,3593.91,2,0,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,39,0,1,0,39640.08,3303.34,0,0.275,691,16322.23,...,36,566.22,14,2,23748.10,5801.45,1,0,4,0
19996,66,0,1,0,32062.90,2671.91,0,0.367,758,16697.34,...,36,553.71,9,8,49929.65,40901.31,3,0,3,1
19997,65,0,2,2,18642.02,1553.50,3,0.106,751,23924.78,...,36,772.66,8,3,13137.57,5075.67,1,0,2,1
19998,35,1,1,2,22181.39,1848.45,1,0.275,646,16920.13,...,36,595.36,16,5,19580.82,3876.16,4,0,5,1


In [41]:
# Encoding Categorical Columns
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
categorical_columns = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
for col in categorical_columns:
    df[col] = le.fit_transform(df[col])

    df.head()

In [42]:
# Seperate Input and Output
indep_X = df.drop("loan_paid_back", axis = 1)
dep_Y = df["loan_paid_back"]

In [43]:
# Train,Test and Split

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 42)


In [44]:
# Feature Scaling

from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.fit(X_test)


# Feature Selection

In [45]:
# i choose Select KBest feature selection:

from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(score_func = f_classif, k = 5)
X_selected = selector.fit_transform(indep_X, dep_Y)

selected_features = indep_X.columns[selector.get_support()]
print("Selected Feature:")
print(selected_features)

Selected Feature:
Index(['employment_status', 'debt_to_income_ratio', 'credit_score',
       'interest_rate', 'grade_subgrade'],
      dtype='str')


In [46]:
indep_X = pd.DataFrame(X_selected, columns = selected_features)
indep_X.head()

,employment_status,debt_to_income_ratio,credit_score,interest_rate,grade_subgrade
0,0.0,0.074,743.0,13.39,9.0
1,0.0,0.219,531.0,17.81,25.0
2,0.0,0.234,779.0,9.53,8.0
3,0.0,0.264,809.0,7.99,4.0
4,0.0,0.260,663.0,15.20,19.0


In [47]:
y = df['loan_paid_back']

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(indep_X, dep_Y, test_size = 0.2, random_state = 42)

In [48]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [49]:
models = {"Logisitic Regression": LogisticRegression(random_state = 42),
         "Decision Tree": DecisionTreeClassifier(random_state = 42),
         "Random Forest": RandomForestClassifier(random_state = 42),
         "KNN": KNeighborsClassifier(),
         "SVM": SVC(),
         "Naive Bayes": GaussianNB()
        }

In [50]:
def evaluate_model(models, X_train, X_test, y_train, y_test):
    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test,y_pred)

        results[name] = accuracy

        print("="*50)
        print("Model:", name)
        print("Accuracy:", accuracy)
        print("="*50)

        print("Classification Report")
        print(classification_report(y_test,y_pred))

        print("Confusion Matrix")
        print(confusion_matrix(y_test,y_pred))

    return results

In [51]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(16000, 5)
(4000, 5)
(16000,)
(4000,)


In [52]:
results = evaluate_model(models, X_train, X_test, y_train, y_test)

Model: Logisitic Regression
Accuracy: 0.88275
Classification Report
              precision    recall  f1-score   support

           0       0.81      0.55      0.66       818
           1       0.89      0.97      0.93      3182

    accuracy                           0.88      4000
   macro avg       0.85      0.76      0.79      4000
weighted avg       0.88      0.88      0.87      4000

Confusion Matrix
[[ 452  366]
 [ 103 3079]]
Model: Decision Tree
Accuracy: 0.8365
Classification Report
              precision    recall  f1-score   support

           0       0.60      0.61      0.60       818
           1       0.90      0.90      0.90      3182

    accuracy                           0.84      4000
   macro avg       0.75      0.75      0.75      4000
weighted avg       0.84      0.84      0.84      4000

Confusion Matrix
[[ 496  322]
 [ 332 2850]]
Model: Random Forest
Accuracy: 0.8945
Classification Report
              precision    recall  f1-score   support

           0   

C:\Anaconda3\envs\aiml\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Anaconda3\envs\aiml\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Anaconda3\envs\aiml\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [53]:
accuracy_df = pd.DataFrame(results.items(), columns =["Model", "Accuracy"])

accuracy_df = accuracy_df.sort_values(by = "Accuracy", ascending =False)

accuracy_df

,Model,Accuracy
2,Random Forest,0.89450
0,Logisitic Regression,0.88275
5,Naive Bayes,0.86525
3,KNN,0.85800
1,Decision Tree,0.83650
4,SVM,0.79550
